# 08 — Before/after picture: what calibration covers, what fine-tuning covers

**What this runs (GPU only, ~35 min on A100):**

- **Part A** retrains the fine-tuned model of the sealed mCellSeg test with the same recipe on the same 79
  calibration images (the scored weights were not kept). Its weights go to Drive this time.
- **Part B** reads the 90 test images with it (a reproducibility check against the scored readings, not a new
  score) and saves full-size coverage masks of the four images the committed rule picks.

**Before running:** upload `confluency_v4_bundle.zip` to `MyDrive/cultureqc/staged/`.

**Runtime:** A100 with **High-RAM** if offered.

**Bring back:** `MyDrive/cultureqc/staged/confluency_v4_results.zip` (a few MB). Leave `mcellseg_ftF_r2.weights`
(about 1 GB) on Drive.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, time, subprocess, json, hashlib, zipfile
DRIVE_ROOT = '/content/drive/MyDrive/cultureqc'
STAGE_DIR = f'{DRIVE_ROOT}/staged'
BUNDLE = f'{STAGE_DIR}/confluency_v4_bundle.zip'
RESULTS_ZIP = f'{STAGE_DIR}/confluency_v4_results.zip'
EXPECTED_COMMIT = '5e24fa4'        # the pre-registration commit the bundle was made from
REPO = '/content/cultureqc'
assert os.path.exists(BUNDLE), f'upload the bundle first: {BUNDLE}'
print(f'{BUNDLE}: {os.path.getsize(BUNDLE)/1e6:.0f} MB')

In [ ]:
t0 = time.time()
shutil.rmtree(REPO, ignore_errors=True)
shutil.copy(BUNDLE, '/content/bundle.zip')                 # one sequential read from Drive
!unzip -q -o /content/bundle.zip -d $REPO
os.remove('/content/bundle.zip')
%cd $REPO
commit = open('COMMIT').read().strip()
print('bundle commit', commit, f'— unpacked in {time.time()-t0:.0f} s')
assert commit.startswith(EXPECTED_COMMIT), (commit, EXPECTED_COMMIT)

In [ ]:
!pip install -q cellpose==4.2.1.1 tifffile scikit-image opencv-python-headless pyyaml
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU (A100)'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Same model as the product: the Cellpose-SAM weights Colab downloads must hash to the ones every shipped
# record names (demo/examples/examples.json, model_weights_hash.seg).
from cellpose import models
_ = models.CellposeModel(gpu=True)
w = os.path.join(str(models.MODEL_DIR), 'cpsam_v2')
h = hashlib.sha256(open(w, 'rb').read()).hexdigest()
print('cpsam_v2', h)
assert h == '0f1cc3f7ecdd8a037a57c6c48d9d8921391be4cbce3fa9f13c3e3a2e1253c667', 'not the shipped weights: stop and tell me'
# The pre-registered files are byte-identical to the pre-registration commit
SEALED = {
 "scripts/confluency_mcellseg.py": "fd236fe2d51f8d1cd777450e4c6e257b6b44fa1a768428859c7f20c11df9f084",
 "results/confluency_mcellseg_split.json": "84f6fb7e4eb99bdefc5f07ce7aeace8f7d0bfb68ff8f4bd7422c8348ef029c15",
 "results/confluency_mcellseg.json": "f5f74a85a430ee34cd61ea4458c7da6990525639999cfda487bcbebb02e8e4ba",
 "results/confluency_mcellseg.csv": "81db44c35ab26cd9043dd26339862a62b2fd66d3289912ce6ab747c34c8285b9",
 "results/confluency_mcellseg_curves.json": "5cd6ebf070a7e0cd9844ded142b213f5f31447ffcc75873c631a2b3474d6a09a",
 "scripts/confluency_finetune_figure.py": "3e9d01b63df51375cc53030b0274bee2bef601ff1585bdb795a002d7d10276f3"
}
for f, s in SEALED.items():
    assert hashlib.sha256(open(f, 'rb').read()).hexdigest() == s, f
print('pre-registered files match', EXPECTED_COMMIT)

In [ ]:
# mCellSeg (Alam et al. 2026, CC BY 4.0) straight from Zenodo to local disk, MD5 from the record
t0 = time.time()
os.makedirs('/content/dl', exist_ok=True)
!curl -sL -o /content/dl/mCellSeg.zip "https://zenodo.org/api/records/20174259/files/mCellSeg.zip/content"
md5 = hashlib.md5(open('/content/dl/mCellSeg.zip', 'rb').read()).hexdigest()
print('md5', md5)
assert md5 == 'ed06d6e4c10e93b81703984cef852246'
os.makedirs('data/sources/mcellseg', exist_ok=True)
!unzip -q -o /content/dl/mCellSeg.zip -d data/sources/mcellseg
os.remove('/content/dl/mCellSeg.zip')
!ls data/sources/mcellseg/mCellSeg/labeled/images | wc -l
print(f'{time.time()-t0:.0f} s')

In [ ]:
# Masks only: the split must match the committed one
!python scripts/confluency_mcellseg.py count

### ⏸ Checkpoint
The last line above must say **`split matches the committed results/confluency_mcellseg_split.json`**, and the
cell before it **`pre-registered files match`**. If either fails, stop and send me the output.

In [ ]:
def save_results(label):
    # one small zip to Drive: reading file + run manifests + logs (no maps, no weights)
    keep = ['results/confluency_finetune_figure_curves.json', 'cache/finetune/mcellseg_ftF_r2/manifest.json']
    keep += [os.path.join('cache/figure_masks', f) for f in sorted(os.listdir('cache/figure_masks'))] if os.path.isdir('cache/figure_masks') else []
    keep += [os.path.join('logs', f) for f in sorted(os.listdir('logs'))] if os.path.isdir('logs') else []
    local = '/content/confluency_v4_results.zip'
    with zipfile.ZipFile(local, 'w', zipfile.ZIP_DEFLATED) as z:
        z.writestr('COMMIT', commit)
        for p in keep:
            if os.path.exists(p):
                z.write(p)
    shutil.copy(local, RESULTS_ZIP)
    print(label, '->', RESULTS_ZIP, f'{os.path.getsize(local)/1e6:.1f} MB')
    with zipfile.ZipFile(local) as z:
        for n in z.namelist(): print('  ', n)

os.makedirs('logs', exist_ok=True)
def run(cmd, log):
    # streams the output (minus progress bars) and keeps a full log; stops the cell if the step fails
    t0 = time.time()
    r = subprocess.run(['bash', '-c', f'set -o pipefail; {cmd} 2>&1 | tee -a logs/{log}.log | (grep -v -E "it/s]|s/it]" || true)'])
    print(f'[{log}] {(time.time()-t0)/60:.1f} min, exit {r.returncode}')
    if r.returncode != 0:
        save_results(f'{log} FAILED (partial results)')
        raise RuntimeError(f'{log} failed — send me logs/{log}.log (it is in the results zip)')

## Part A — retrain with the same recipe (~25 min on A100)

The zip is saved before the 1 GB weights copy and again after it, so a failed copy cannot lose the results.

In [ ]:
run('python scripts/confluency_finetune_figure.py train', 'figure_train')
save_results('Part A (before the weights copy)')
man = json.load(open('cache/finetune/mcellseg_ftF_r2/manifest.json'))
print(man['n'], 'images; weights sha256', man['weights_sha256'])
shutil.copy(man['weights'], f'{STAGE_DIR}/mcellseg_ftF_r2.weights')
assert hashlib.sha256(open(f'{STAGE_DIR}/mcellseg_ftF_r2.weights', 'rb').read()).hexdigest() == man['weights_sha256']
print('weights on Drive:', f'{STAGE_DIR}/mcellseg_ftF_r2.weights')
save_results('Part A')

## Part B — reproducibility readings and the four masks (~5 min)

In [ ]:
run('python scripts/confluency_finetune_figure.py readings', 'figure_readings')
save_results('Part B')

## Done
Download `MyDrive/cultureqc/staged/confluency_v4_results.zip` and tell me where it is (Downloads is fine). If your browser saves it as `confluency_v4_results(1).zip`, tell me that name.